# 이커머스 구매 퍼널 분석

## 분석 개요
- 목표: 가상의 이커머스 상품의 채널별 구매 전환율 차이를 검정하고, 개선 우선순위를 도출
- 비교군: 모바일 / PC
- 종속변수: 퍼널 단계별 전환 여부 (전환=1, 이탈=0)
- 표본: 10,000 세션 (시뮬레이션)
- 검정 방법: 카이제곱 독립성 검정 (α = 0.05)

## 분석 순서
1. 데이터 생성
2. 퍼널 시각화
3. 카이제곱 독립성 검정
4. 개선 효과 추정 및 제안


In [92]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import plotly.express as px


In [ ]:
# runtime parameter 설정

from matplotlib import rcParams

rcParams['font.family'] = "Malgun Gothic"
rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", font = "Malgun Gothic")

## 데이터 생성
- 1만개의 데이터를 랜덤하게 생성
- 모바일 사용이 더 많은 제품이라는 가정 하에 샘플 생성 (p = [0.65, 0.35])

In [ ]:
np.random.seed(42)

# 유저 100명, 유저당 세션 수 1~ 10개 
n_users = 100 
sessions_per_user = np.random.randint(1,11, size=n_users)

records = []

for uid in range(n_users):
    for sid in range(1, sessions_per_user[uid] + 1):
        channel = np.random.choice(['mobile', 'pc'], p = [0.65, 0.35])
        records.append({'user_id': uid + 1, 'session_id': sid, 'channel':channel, 'event_type': 'page_view'})

        p_view = 0.62 if channel == 'mobile' else 0.75
        
        if np.random.random() < p_view:
            records.append({'user_id': uid+1, 'session_id' : sid, 'channel': channel, 'event_type' : 'product_view'})
        else:
            continue

        p_cart = 0.35 if channel == 'mobile' else 0.50
        if np.random.random() < p_cart:
            records.append({'user_id': uid + 1, 'session_id': sid, 'channel': channel, 'event_type': 'add_to_cart'})
        else:
            continue

        p_purchase = 0.3 if channel == 'mobile' else 0.55
        if np.random.random() < p_purchase:
            records.append({'user_id':uid+1, 'session_id':sid, 'channel':channel,'event_type':'purchase'})


        
df = pd.DataFrame(records)

df


In [ ]:
# 데이터 검수
print(df.shape)
print(df['event_type'].value_counts())
print(df['channel'].value_counts())

## 시각화

In [ ]:
# 이탈률 확인

stages = ['page_view', 'product_view', 'add_to_cart', 'purchase']
stage_counts = df['event_type'].value_counts()[stages]

print(stage_counts)
print()

for i in range(1, len(stages)):
    churn = (1- stage_counts[stages[i]] / stage_counts[stages[i-1]]) * 100
    print(f' {stages[i-1]} -> {stages[i]} 이탈률%: -{churn:.1f}%')

event_type
page_view       591
product_view    401
add_to_cart     146
purchase         70
Name: count, dtype: int64

 page_view -> product_view 이탈률%: -32.1%
 product_view -> add_to_cart 이탈률%: -63.6%
 add_to_cart -> purchase 이탈률%: -52.1%


In [ ]:
fig = px.funnel(
    x=stage_counts.values,
    y=['방문', '상품조회', '장바구니', '구매완료'], 
        color_discrete_sequence= px.colors.sequential.Sunset_r

)

# 이탈률 데이터 레이블 만들기
counts = stage_counts.values
labels = ['']
for i in range(1, len(counts)):
    churn_rate = (1 - counts[i] / counts[i-1]) * 100
    labels.append(f'-{churn_rate:.1f}%')

fig.update_traces(text=labels, textposition='inside')
fig.update_layout(yaxis_title=None, 
                  title = {'text': '구매퍼널', 
                           'x': 0.5, 
                           'font' : {'size' : 18, 'weight' : 'bold'}})

fig.show()


52.05479452054795
